#### PERFORM STREAMING 
- `GREEN-TAXI`

In [0]:
SOURCE_PATH = '/Volumes/nyctaxi/landing/green_taxi/2026/*'

green_taxi_streaming_df = (spark.readStream.format('cloudFiles')
                                 .option('cloudFiles.format', 'parquet')
                                 .option("cloudFiles.schemaLocation", "/Volumes/nyctaxi/bronze/schemaLogs/green_taxi")
                                 .option('cloudFiles.schemaEvolutionMode', 'addNewColumns')
                                 .load(SOURCE_PATH)
                        )

transformed_df = (
    green_taxi_streaming_df
    .selectExpr(
        "*",
        "_metadata.file_path as file_path",
        "_metadata.file_name as file_name",
        "_metadata.file_modification_time as load_timestamp"
    )
)

In [0]:
(transformed_df.writeStream
    .format('delta')
    .outputMode('append')
    .option("checkpointLocation", "/nyctaxi/bronze/checkpoints/green_taxi")
    .table('NYCTAXI.BRONZE.GREEN_TAXI')
)